# Exploratory Data Analysis — New Zealand University Enrollments

This notebook provides a univariate EDA of New Zealand bachelor's-degree enrolments by broad field of study, 2016–2025.

**Data source:** `data/clean/NZ_bachelors_enrollments_2016_2025.csv`  
Derived from MoE FOS_ENR.2 (all tertiary providers). Based on the 2025 FOS_ENR.3 sub-sector split, universities account for approximately **77 %** of bachelor enrolments across all providers — this file therefore slightly overstates university-only figures and the scale limitation should be noted in any comparisons with AUS (universities only) or UK (universities only).

**Role in the JRGS project:** New Zealand is used as a **second control group** alongside the United Kingdom in the difference-in-differences (DiD) analysis of the Job-Ready Graduates Scheme (JRG, 2021). This EDA characterises the NZ enrolment baseline and assesses suitability for the control-group role.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

# Resolve dataset path robustly
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
dataset_path = None
for root in candidate_roots:
    candidate = root / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv'
    if candidate.exists():
        dataset_path = candidate
        break

if dataset_path is None:
    raise FileNotFoundError(f'Dataset not found from working directory: {Path.cwd()}')

df = pd.read_csv(dataset_path)

print(f'Loaded {dataset_path.name}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
print('Columns:', df.columns.tolist())
df.head()

## 1. Inspect Data Structure

Review column names, data types, shape, and coverage. The NZ file is already in **long format** (one row per field × year) unlike the wide-format AUS/UK files.

In [ ]:
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes.to_frame('dtype'))
print(f'\nShape: {df.shape}')
print(f'Fields: {df["field_of_study"].nunique()}')
print(f'Years covered: {sorted(df["year"].unique())}')
print('\nFirst 10 rows:')
df.head(10)

## 2. Clean Missing Values and Duplicates

Check for null values and duplicate rows, then create a cleaned working copy.

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]

print('Missing values by column:')
if missing_summary.empty:
    print('  No missing values found.')
else:
    print(missing_summary.to_frame('missing_count'))

duplicate_count = df.duplicated().sum()
print(f'\nDuplicate rows: {duplicate_count}')

df_clean = df.drop_duplicates().copy()

# Exclude the 'Total (all fields)' aggregate row for category-level analysis
df_fields = df_clean[df_clean['field_of_study'] != 'Total (all fields)'].copy()
df_total  = df_clean[df_clean['field_of_study'] == 'Total (all fields)'].copy()

print(f'Shape after cleaning: {df_clean.shape}')
print(f'Field rows (excl. total): {len(df_fields)}')

## 3. Compute Summary Statistics

Descriptive statistics for total and domestic/international breakdown.

In [ ]:
print('=== Summary statistics (by enrolment column) ===')
print(df_fields[['domestic_bachelors', 'international_bachelors', 'total_bachelors']].describe().T)

# Annual totals
annual = df_fields.groupby('year')[['domestic_bachelors', 'international_bachelors', 'total_bachelors']].sum()
print('\n=== Annual totals across all fields ===')
print(annual)

# Field-level change: first vs last available year
yr_min = df_fields['year'].min()
yr_max = df_fields['year'].max()
first = df_fields[df_fields['year'] == yr_min].set_index('field_of_study')[['total_bachelors']].rename(columns={'total_bachelors': str(yr_min)})
last  = df_fields[df_fields['year'] == yr_max].set_index('field_of_study')[['total_bachelors']].rename(columns={'total_bachelors': str(yr_max)})
change = first.join(last)
change['Absolute Change'] = change[str(yr_max)] - change[str(yr_min)]
change['Percent Change']  = (change['Absolute Change'] / change[str(yr_min)] * 100).round(2)
print(f'\n=== Change {yr_min}→{yr_max} ===')
print(change.sort_values('Absolute Change', ascending=False).to_string())

## 4. Visualize Enrollment Trends

Plot total and field-level enrolment over time. Include a vertical line at 2021 (JRG policy start in AUS — NZ is the control group, so NZ trends post-2021 inform the counterfactual).

In [ ]:
# --- Total enrolments over time ---
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_total, x='year', y='total_bachelors', marker='o', linewidth=2.5)
plt.axvline(2021, color='red', linestyle='--', linewidth=1.2, label='JRG implementation (AUS 2021)')
plt.title('Total NZ Bachelor Enrolments by Year (All Providers)')
plt.ylabel('Students')
plt.legend()
plt.tight_layout()
plt.show()

# --- By field of study ---
plt.figure(figsize=(12, 7))
sns.lineplot(data=df_fields, x='year', y='total_bachelors', hue='field_of_study', marker='o')
plt.axvline(2021, color='red', linestyle='--', linewidth=1.2)
plt.title('NZ Enrolment Trends by Field (Excl. Total)')
plt.ylabel('Students')
plt.legend(title='Field', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# --- Domestic vs international split ---
split = df_total.melt(id_vars='year', value_vars=['domestic_bachelors', 'international_bachelors'],
                      var_name='Student Type', value_name='Enrolments')
split['Student Type'] = split['Student Type'].str.replace('_bachelors', '', regex=False)
plt.figure(figsize=(10, 5))
sns.lineplot(data=split, x='year', y='Enrolments', hue='Student Type', marker='o', linewidth=2)
plt.axvline(2021, color='red', linestyle='--', linewidth=1.2, label='JRG (AUS)')
plt.title('NZ Domestic vs International Bachelor Enrolments')
plt.tight_layout()
plt.show()

# --- Tukey box plot: enrolment distribution by field ---
latest_year = df_fields['year'].max()
order = (df_fields[df_fields['year'] == latest_year]
         .sort_values('total_bachelors', ascending=False)['field_of_study'].tolist())
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_fields, x='total_bachelors', y='field_of_study', order=order, whis=1.5, color='skyblue')
sns.stripplot(data=df_fields, x='total_bachelors', y='field_of_study', order=order,
              color='black', alpha=0.55, size=4)
plt.title('Tukey Box Plot of NZ Enrolments by Field')
plt.xlabel('Enrolments')
plt.tight_layout()
plt.show()

## 5. Explore Correlations Across Years

Check how yearly enrolment patterns move together.

In [ ]:
year_pivot = df_fields.pivot_table(index='field_of_study', columns='year', values='total_bachelors')
corr_matrix = year_pivot.T.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(year_pivot.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Between NZ Enrolment Years (by field)')
plt.tight_layout()
plt.show()

## 6. Identify Fast-Growing and Unusual Fields

Measure field-level growth and flag outliers.

In [ ]:
yr_min = df_fields['year'].min()
yr_max = df_fields['year'].max()

growth = (df_fields[df_fields['year'].isin([yr_min, yr_max])]
          .pivot_table(index='field_of_study', columns='year', values='total_bachelors')
          .rename(columns={yr_min: 'start', yr_max: 'end'})
          .assign(**{
              'Absolute Change': lambda d: d['end'] - d['start'],
              'Percent Change':  lambda d: (d['end'] - d['start']) / d['start'] * 100,
          }))

q1, q3 = growth['Absolute Change'].quantile([0.25, 0.75])
iqr = q3 - q1
growth['Potential Outlier'] = ~growth['Absolute Change'].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print(growth.sort_values('Absolute Change', ascending=False).to_string())

plt.figure(figsize=(12, 6))
sns.barplot(data=growth.reset_index().sort_values('Absolute Change', ascending=False),
            x='Absolute Change', y='field_of_study', palette='magma', hue='field_of_study', legend=False)
plt.title(f'NZ Enrolment Change {yr_min}–{yr_max}')
plt.tight_layout()
plt.show()

## 7. Individual Tukey Box Plots by Field

In [ ]:
fields = df_fields['field_of_study'].unique().tolist()
n_cols = 2
n_rows = (len(fields) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, field in zip(axes, fields):
    values = df_fields.loc[df_fields['field_of_study'] == field, 'total_bachelors'].dropna()
    sns.boxplot(x=values, ax=ax, whis=1.5, color='skyblue')
    sns.stripplot(x=values, ax=ax, color='black', alpha=0.6, size=5)
    ax.set_title(field, fontsize=9)
    ax.set_xlabel('Enrolments')
    ax.set_yticks([])

for ax in axes[len(fields):]:
    ax.set_visible(False)

plt.suptitle('Independent Tukey Box Plots — NZ Fields of Study', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 8. Additional Statistical Checks

Tests for correlation, nonlinearity, log-transform suitability, Simpson's paradox, and subgroup heterogeneity.

In [ ]:
try:
    from scipy import stats
except ImportError:
    stats = None
    print('scipy not installed; inferential tests skipped.')

# --- Simpson's Paradox check ---
agg = df_fields.groupby('year')['total_bachelors'].sum().reset_index()
if stats:
    r_overall, p_overall = stats.pearsonr(agg['year'], agg['total_bachelors'])
    print(f'Overall r(Year, Total Enrolments) = {r_overall:.3f}  (p={p_overall:.4f})')

    cat_trends = []
    for field, g in df_fields.groupby('field_of_study'):
        if len(g) >= 4:
            r, p = stats.pearsonr(g['year'], g['total_bachelors'])
            cat_trends.append({'field_of_study': field, 'r': r, 'p': p})
    trends = pd.DataFrame(cat_trends).set_index('field_of_study').sort_values('r')
    neg = int((trends['r'] < 0).sum())
    print(f'Within-field negative trends: {neg}/{len(trends)}')
    if neg > 0 and r_overall > 0:
        print("Simpson's Paradox: aggregate POSITIVE but some fields NEGATIVE")

# --- Log-transform skewness ---
if stats:
    sk_raw = stats.skew(df_fields['total_bachelors'].dropna())
    sk_log = stats.skew(np.log1p(df_fields['total_bachelors'].dropna()))
    print(f'Skewness: raw={sk_raw:.3f}  |  log(1+x)={sk_log:.3f}')

# --- Three-panel figure ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('NZ Enrolments — Data Characteristics & First-Order Effects', fontsize=13, fontweight='bold')

if 'trends' in dir():
    colors_bar = ['#e74c3c' if r < 0 else '#27ae60' for r in trends['r']]
    axes[0].barh(trends.index, trends['r'], color=colors_bar)
    axes[0].axvline(r_overall, color='navy', ls='--', lw=2, label=f'Aggregate r={r_overall:.2f}')
    axes[0].set_xlabel('Pearson r  (Year vs Enrolments)')
    axes[0].set_title('A. Per-field vs aggregate trend')
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis='y', labelsize=7)

axes[1].hist(df_fields['total_bachelors'].dropna(), bins=25, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Enrolments (raw)')
axes[1].set_title('B. Raw distribution')

axes[2].hist(np.log1p(df_fields['total_bachelors'].dropna()), bins=25, color='darkorange', edgecolor='white')
axes[2].set_xlabel('log(1 + Enrolments)')
axes[2].set_title('C. Log-transformed distribution')

plt.tight_layout()
plt.show()

# --- Heterogeneity tests ---
groups = [g['total_bachelors'].dropna().values
          for _, g in df_fields.groupby('field_of_study') if len(g) > 1]
if stats and len(groups) >= 2:
    lev_stat, lev_p = stats.levene(*groups, center='median')
    f_stat, f_p     = stats.f_oneway(*groups)
    kw_stat, kw_p   = stats.kruskal(*groups)
    print(f"Levene's test: F={lev_stat:.3f}, p={lev_p:.4g}")
    print(f'One-way ANOVA: F={f_stat:.3f}, p={f_p:.4g}')
    print(f'Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4g}')

## Data Characteristics & First-Order Effects

**Variables:** `field_of_study` (12 broad NZSCED fields incl. total), `category_key` (integer mapping to AUS/UK keys), `year` (2016–2025), and three enrolment columns: `domestic_bachelors`, `international_bachelors`, `total_bachelors`.

**Coverage caveat:** FOS_ENR.2 includes ALL tertiary providers (universities + polytechnics + wananga + PTEs). Based on the 2025 FOS_ENR.3 breakdown, universities account for approximately 77 % of bachelor enrolments in total. Field-level ratios range from ~60 % to ~90 %, so direct headcount comparisons with AUS/UK university-only figures require scaling.

**DiD role:** NZ is the second control group. No NZ-specific parallel policy shock occurred during 2021–2024 that would confound the counterfactual. Post-2021 NZ enrolment trends reflect the global post-COVID higher-education environment, net of any AUS-specific JRG effect.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
clean_dir = next((r / 'data' / 'clean' for r in candidate_roots if (r / 'data' / 'clean').exists()), None)
assert clean_dir, 'Cannot find data/clean directory'

df = pd.read_csv(clean_dir / 'NZ_bachelors_enrollments_2016_2025.csv')
df_fields = df[df['field_of_study'] != 'Total (all fields)'].copy()
df_total  = df[df['field_of_study'] == 'Total (all fields)'].copy()

print('=== NZ Enrolments — Variable Summary ===')
print(f'Shape (long, fields only): {df_fields.shape}')
print(f'Fields: {df_fields["field_of_study"].nunique()}')
print(f'Year range: {df_fields["year"].min()}–{df_fields["year"].max()}')
print(f'Total_bachelors range: {df_fields["total_bachelors"].min():,.0f}–{df_fields["total_bachelors"].max():,.0f}')

# Indexed trend (base 2019 = 100) relative to JRG year
base = 2019
base_vals = df_fields[df_fields['year'] == base].set_index('field_of_study')['total_bachelors']
df_indexed = df_fields.copy()
df_indexed['Index'] = df_indexed.apply(
    lambda r: r['total_bachelors'] / base_vals.get(r['field_of_study'], np.nan) * 100, axis=1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NZ Enrolments — Data Characteristics', fontsize=12, fontweight='bold')

total_idx = df_total.copy()
total_idx['Index'] = total_idx['total_bachelors'] / total_idx.loc[total_idx['year'] == base, 'total_bachelors'].values[0] * 100
sns.lineplot(data=total_idx, x='year', y='Index', ax=axes[0], marker='o', linewidth=2.5)
axes[0].axvline(2021, color='red', ls='--', lw=1.5, label='JRG (AUS 2021)')
axes[0].axhline(100, color='grey', ls=':', lw=1)
axes[0].set_title(f'A. NZ Total Enrolments Indexed ({base}=100)')
axes[0].set_ylabel('Index')
axes[0].legend(fontsize=8)

axes[1].hist(np.log1p(df_fields['total_bachelors'].dropna()), bins=25,
             color='steelblue', edgecolor='white')
sk_log = stats.skew(np.log1p(df_fields['total_bachelors'].dropna()))
axes[1].set_xlabel('log(1 + total_bachelors)')
axes[1].set_title(f'B. Log-transformed distribution (skewness={sk_log:.2f})')

plt.tight_layout()
plt.show()

# Empirical CDF
fig2, ax2 = plt.subplots(figsize=(8, 4))
sns.ecdfplot(df_fields['total_bachelors'], ax=ax2, color='darkorange')
ax2.axvline(df_fields['total_bachelors'].median(), color='navy', ls='--', lw=1.5,
            label=f"Median={df_fields['total_bachelors'].median():,.0f}")
ax2.set_xlabel('Enrolments')
ax2.set_title('Empirical CDF — NZ Field Enrolments')
ax2.legend()
plt.tight_layout()
plt.show()

### What Is Learned

1. **Variable characteristics:** TODO — complete after running analysis.
2. **Data cleaning outcome:** TODO
3. **Distribution and transformation:** TODO
4. **Simpson's Paradox check:** TODO
5. **DiD modelling implications:** TODO — specifically, does NZ show a stable growth trajectory post-2021 consistent with a valid control group?